# C1 · Duplicate rows: the OOB error (and the cross-validation) say 100%

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Build a noisy dataset in which every row appears **twice** (or almost: with a tiny measurement noise) and show that both the OOB accuracy and a random cross-validation become optimistic, because the copy of an example that is "outside the sample" is often inside. Fix it with
validation **by group** (`GroupKFold`) and with the removal of the duplicates.

## Dataset

`make_classification(1000, 10 features, 2 informative, flip_y=0.4)` (very noisy: the true generalization is about 0.75). `X_d, y_d, group` is the same dataset with every row repeated twice (`group` = index of the original row); `X_q` is the version with the copy perturbed by a noise of 1e-3.

## Educational purpose

Recognize when an "internal" error (OOB or K-fold) is optimistic: it happens every time non-independent rows (duplicates, repeated measurements of the same subject, nearly identical images) end up in different folds. It is the group leakage of module 3, inside the forest.

## Task

1. Train `RandomForestClassifier(200, oob_score=True, random_state=0, n_jobs=-1)` on the **clean** dataset (`X, y`) and save `clean_oob`; on the dataset with exact duplicates `X_d, y_d` save `dup_oob` and `near_oob` (with `X_q`).
2. Cross-validation: `random_cv` (`StratifiedKFold(5, shuffle=True, random_state=0)` on the duplicates) and `group_cv` (`GroupKFold(5)` with `groups=group`, same model).
3. Save `oob_inflation = dup_oob − clean_oob`, `cv_inflation = random_cv − group_cv`.
4. Fix it: remove the exact duplicates (`np.unique` on the whole rows or `pd.DataFrame(...).drop_duplicates()`) and save `oob_after_dedup`. Why can't you remove the near-duplicates (`X_q`) with `drop_duplicates`? Save `n_rows_after_near_dedup` (rows left in `X_q` after `drop_duplicates`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score

X, y = make_classification(1000, n_features=10, n_informative=2, n_redundant=0, flip_y=0.4, random_state=3)
X_d, y_d = np.r_[X, X], np.r_[y, y]
group = np.r_[np.arange(1000), np.arange(1000)]
X_q = np.r_[X, X + np.random.default_rng(0).normal(0, 1e-3, X.shape)]
rf = lambda: RandomForestClassifier(200, oob_score=True, random_state=0, n_jobs=-1)

## Your work

In [ ]:
clean_oob = None
dup_oob = None
near_oob = None

In [ ]:
random_cv = None
group_cv = None
oob_inflation = None
cv_inflation = None

In [ ]:
oob_after_dedup = None
n_rows_after_near_dedup = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`pd.DataFrame(np.c_[X_d, y_d]).drop_duplicates().to_numpy()` then separate the columns. `GroupKFold`: `cross_val_score(rf(), X_d, y_d, groups=group, cv=GroupKFold(5)).mean()`. `oob_score_` after `fit`.'):
    check(0.7 < clean_oob < 0.82, "on the clean dataset the OOB is realistic (0.70-0.82)", "RandomForestClassifier(oob_score=True) on X, y")
    check(dup_oob > 0.97 and near_oob > 0.95, "with exact duplicates the OOB goes over 0.97; with near-duplicates 0.95", "the copy of an 'outside' example is inside the bootstrap")
    check(random_cv > 0.9 and 0.7 < group_cv < 0.82, "random cross-validation on the duplicates goes over 0.9; by group it goes back to 0.70-0.82", "GroupKFold(5) with groups=group")
    check(oob_inflation > 0.2 and cv_inflation > 0.15 and close(oob_inflation, dup_oob - clean_oob), "inflation of over 0.2 for the OOB and over 0.15 for the cross-validation", "differences with the correct estimate")
    check(abs(oob_after_dedup - clean_oob) < 0.02 and n_rows_after_near_dedup == 2000, "after removing the duplicates the OOB goes back to the clean one; the near-duplicates are not removed (2000 rows stay)", "drop_duplicates compares identical values")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_duplicate_rows_and_optimistic_oob.ipynb`. Open it after you pass the check, or after you have really tried.